# Extract images listed in `val.txt` from `line_images` folder

This notebook:

1. Reads a `val.txt` file where each line begins with a *line address* followed by annotation.
2. Extracts the address (first whitespace-separated token on each line).
3. Searches for the corresponding image file inside a `line_images` folder.
4. Copies found images into an output folder (`selected_lines`).

You can change the paths in the **Parameters** cell below and run all cells.

In [3]:
# Parameters - change these paths if needed
VAL_TXT_PATH = 'val.txt'        # path to val.txt
LINE_IMAGES_DIR = 'line_images' # folder containing line images
OUTPUT_DIR = 'selected_lines'   # folder to create with extracted images
# If filenames in val.txt include directory parts, they'll be handled. Default behavior extracts first whitespace-separated token as the address.


In [4]:
import os
from pathlib import Path
import shutil
from collections import defaultdict

def extract_addresses(val_txt_path):
    addresses = []
    with open(val_txt_path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            # Address is the first whitespace-separated token (as requested)
            addr = line.split(None, 1)[0]
            addresses.append(addr)
    return addresses

def find_image_for_address(address, images_dir):
    """Try resolving the address to a file in images_dir.
    Strategies:
    - If address is a path, try that path relative to images_dir and absolute.
    - If address has an extension, try direct match by basename.
    - If address has no extension, try common image extensions.
    - If multiple matches exist, return the first found match.
    """
    p = Path(address)
    images_dir = Path(images_dir)
    # 1) If address is an absolute path and exists, return it
    if p.is_absolute() and p.exists():
        return p
    # 2) If address contains directories, treat relative to images_dir
    candidate = images_dir.joinpath(p)
    if candidate.exists():
        return candidate
    # 3) Try matching by basename (with and without extension)
    name = p.name
    # If name already has an extension, try to find that exact file in images_dir (recursive search)
    if Path(name).suffix:
        for match in images_dir.rglob(name):
            return match
    else:
        # try common extensions
        for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.gif', '.webp']:
            for match in images_dir.rglob(name + ext):
                return match
    # 4) fallback: check if any file contains the name as substring (may be slower)
    for match in images_dir.rglob('*'):
        if match.is_file() and name in match.stem:
            return match
    return None

def copy_selected_images(addresses, images_dir, out_dir):
    images_dir = Path(images_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    missing = []
    duplicates = defaultdict(list)
    
    for addr in addresses:
        found = find_image_for_address(addr, images_dir)
        if found:
            dest = out_dir.joinpath(found.name)
            # handle name collisions by appending a counter
            if dest.exists():
                base = dest.stem
                ext = dest.suffix
                i = 1
                while out_dir.joinpath(f"{base}_{i}{ext}").exists():
                    i += 1
                dest = out_dir.joinpath(f"{base}_{i}{ext}")
            shutil.copy2(found, dest)
            copied.append({'address': addr, 'found_path': str(found), 'copied_to': str(dest)})
            duplicates[found.name].append(addr)
        else:
            missing.append({'address': addr})
    return copied, missing, duplicates

# Example usage (will be executed in the next cell after parameter adjustments)


In [5]:
# Run the extraction process using the parameters above
from pprint import pprint
VAL_TXT_PATH = VAL_TXT_PATH
LINE_IMAGES_DIR = LINE_IMAGES_DIR
OUTPUT_DIR = OUTPUT_DIR

# Basic checks
if not Path(VAL_TXT_PATH).exists():
    raise FileNotFoundError(f"val.txt not found at: {VAL_TXT_PATH}")
if not Path(LINE_IMAGES_DIR).exists():
    raise FileNotFoundError(f"line_images directory not found at: {LINE_IMAGES_DIR}")

addresses = extract_addresses(VAL_TXT_PATH)
print(f"Addresses extracted: {len(addresses)} (showing first 10):\n", addresses[:10])

copied, missing, duplicates = copy_selected_images(addresses, LINE_IMAGES_DIR, OUTPUT_DIR)
print(f"\nCopied {len(copied)} images to '{OUTPUT_DIR}'. Missing: {len(missing)}.")


Addresses extracted: 985 (showing first 10):
 ['line_images/216_nataksamaysarbhasha_100.jpg', 'line_images/52_nataksamaysarbhasha_101_137.jpg', 'line_images/75.jpg', 'line_images/126_pandulipi_8.jpg', 'line_images/30_pandulipi_7.jpg', 'line_images/166_siddhipooja.jpg', 'line_images/13_sidhipooja.jpg', 'line_images/58_pandulipi_8.jpg', 'line_images/47_pandulipi_2.jpg', 'line_images/112_navtatvabhasha.jpg']

Copied 980 images to 'selected_lines'. Missing: 5.


In [6]:
# Show a small table summary of copied and missing items
import pandas as pd
copied_df = pd.DataFrame(copied)
missing_df = pd.DataFrame(missing)
display(copied_df.head(30))
print('\nMissing addresses (first 30):')
display(missing_df.head(30))

# Save CSV reports inside the output folder
outp = Path(OUTPUT_DIR)
copied_df.to_csv(outp / 'copied_report.csv', index=False)
missing_df.to_csv(outp / 'missing_report.csv', index=False)
print(f"\nReports saved to: {outp / 'copied_report.csv'} and {outp / 'missing_report.csv'}")


,address,found_path,copied_to
0,line_images/216_nataksamaysarbhasha_100.jpg,line_images/216_nataksamaysarbhasha_100.jpg,selected_lines/216_nataksamaysarbhasha_100.jpg
1,line_images/52_nataksamaysarbhasha_101_137.jpg,line_images/52_nataksamaysarbhasha_101_137.jpg,selected_lines/52_nataksamaysarbhasha_101_137.jpg
2,line_images/75.jpg,line_images/75.jpg,selected_lines/75.jpg
3,line_images/126_pandulipi_8.jpg,line_images/126_pandulipi_8.jpg,selected_lines/126_pandulipi_8.jpg
4,line_images/30_pandulipi_7.jpg,line_images/30_pandulipi_7.jpg,selected_lines/30_pandulipi_7.jpg
5,line_images/166_siddhipooja.jpg,line_images/166_siddhipooja.jpg,selected_lines/166_siddhipooja.jpg
6,line_images/13_sidhipooja.jpg,line_images/13_sidhipooja.jpg,selected_lines/13_sidhipooja.jpg
7,line_images/58_pandulipi_8.jpg,line_images/58_pandulipi_8.jpg,selected_lines/58_pandulipi_8.jpg
8,line_images/47_pandulipi_2.jpg,line_images/47_pandulipi_2.jpg,selected_lines/47_pandulipi_2.jpg
9,line_images/112_navtatvabhasha.jpg,line_images/112_navtatvabhasha.jpg,selected_lines/112_navtatvabhasha.jpg



Missing addresses (first 30):


,address
0,line_images/653b_nataksamaysarbhasha_100.jpg
1,line_images/465a_nataksamaysarbhasha_100.jpg
2,line_images/534b_nataksamaysarbhasha_100.jpg
3,line_images/535b_nataksamaysarbhasha_100.jpg
4,line_images/404_nataksamaysarbhasha_100.jpg



Reports saved to: selected_lines/copied_report.csv and selected_lines/missing_report.csv


## Notes and troubleshooting

- If many addresses are missing, check the exact format in `val.txt`. The code assumes the address is the first whitespace-separated token on each line.
- If addresses include folders, the notebook will try resolving them relative to `line_images`.
- The search uses recursive glob which may be slower on very large datasets; consider pre-indexing filenames if performance is needed.

If you'd like, I can modify the notebook to treat the address differently (for example: use the whole line up to a `|` separator, or use a different tokenization).